In [1]:
import numpy as np
import pandas as pd
import ollama
from datetime import datetime
from tqdm.auto import tqdm

# Load main dataset

In [2]:
# load dataframe
df_kb = pd.read_csv('../data/data-kb.csv', sep='\t', dtype=str) # they are all strings!
df_kb

,pmid,elocationid,title,journal,year,author,affiliation,abstract
0,40935315,doi: 10.1016/j.expneurol.2025.115457,Longitudinal analysis of age-dependent phenoty...,Experimental neurology,2025,"Aiswaria Lekshmi Kannan, Abhisarika Patnaik, A...","Neuroscience Division, IRCCS San Raffaele Scie...",Mutations in cyclin-dependent kinase-like 5 (C...
1,40935250,doi: 10.1016/j.jad.2025.120270,Adverse childhood experiences and cardiometabo...,Journal of affective disorders,2025,"Alyna Turner, Maya Kuperberg, Hugh McGovern, A...","Deakin University, School of Medicine, IMPACT,...",Bipolar disorder is associated with increased ...
2,40935239,pii: S0306-4522(25)00924-8,Endocrine circuitry in autism spectrum disorde...,Neuroscience,2025,"Maria Angelopoulou, Panayiotis Siaperas, Saran...","Department of Pediatric Endocrinology, Univers...",The increasing global prevalence of Autism Spe...
3,40934839,doi: 10.1016/j.seizure.2025.08.030,Tough to treat: What we know about managing PC...,Seizure,2025,"Aleksandra Tobiasz, Monika Hager, Julia Dębows...","Students' Scientific Society, Department of Pe...","PCDH19-related epilepsy is a rare, X-linked de..."
4,40934838,doi: 10.1016/j.seizure.2025.09.004,Characterizing CHD2-associated epilepsy: A mul...,Seizure,2025,"Anabel G Puri, Sean Woods, John M Schreiber, C...","Department of Pediatrics, Division of Pediatri...",CHD2 variants have been implicated in a spectr...
...,...,...,...,...,...,...,...,...
2995,40242676,pii: e80721,Current Clinical Practices for Gaming Disorder...,Cureus,2025,"Masaru Tateno, Yukie Tateno, Tomohiro Shirasak...","Pediatrics, Furuta Pediatric Clinic, Sapporo, ...",Gaming is a popular leisure activity among chi...
2996,40242361,pii: e102012,"Evaluating the feasibility, safety and efficac...",General psychiatry,2025,"Hangyu Tan, Mingyu Xu, Tai Ren, Lin Deng, Ling...",Ministry of Education - Shanghai Key Laborator...,NaN
2997,40242083,pii: 100765,Neuropsychiatric disorders in Chinese pediatri...,Epilepsy & behavior reports,2025,"Jie Fu, Qinrui Li, Genfu Zhang, Zhixian Yang, ...","Department of Pediatrics, Peking University Pe...",Tuberous sclerosis complex (TSC) is an autosom...
2998,40241535,pii: 24121,[The transition from child to adult health car...,Lakartidningen,2025,"Sven Bölte, Tatja Hirvikoski, Ulf Jonsson, Joh...",NaN,The transition from child to adult health care...


# Sample PMIDs, initialize ground truth dataframe

In [3]:
# set sample size
sample_size = 100
print(sample_size)

100


In [4]:
# set number of questions per PMID
num_questions_per_pmid = 5
print(num_questions_per_pmid)

5


In [5]:
# sample PMIDs
missing_abstract = df_kb['abstract'].isna()
sampled_pmids = df_kb['pmid'][missing_abstract==False].sample(n=sample_size, \
random_state=824) # sample from those with abstract
sampled_pmids.values

array(['40340942', '40817004', '40657596', '40731376', '40449672',
       '40844520', '40362647', '40402336', '40883686', '40879423',
       '40454250', '40355280', '40800991', '40448827', '40248830',
       '40708961', '40317349', '40756620', '40620694', '40776983',
       '40406044', '40827489', '40820227', '40490763', '40603888',
       '40340572', '40473417', '40873538', '40775643', '40279814',
       '40714295', '40563066', '40466582', '40874111', '40287634',
       '40762107', '40399107', '40634286', '40460647', '40375705',
       '40324672', '40911257', '40855817', '40366555', '40697049',
       '40419562', '40425098', '40257807', '40478461', '40766520',
       '40766880', '40495546', '40760909', '40415354', '40622845',
       '40398411', '40895831', '40455129', '40679749', '40381091',
       '40549641', '40327194', '40815586', '40426370', '40685559',
       '40777446', '40385454', '40271992', '40630348', '40801633',
       '40917941', '40749089', '40491279', '40672343', '405172

In [6]:
# initialize ground truth dataframe
df_synth = pd.DataFrame({'pmid' : sorted(sampled_pmids.to_list()*num_questions_per_pmid), 
                      'ollama_seed' : [i for i in range(num_questions_per_pmid)]*sample_size})
df_synth = df_synth.merge(df_kb, on=['pmid'], how='left')[['pmid', 'ollama_seed', 'abstract']]
df_synth # has seed for reproducibility

,pmid,ollama_seed,abstract
0,40248830,0,As social relationships are intertwined with m...
1,40248830,1,As social relationships are intertwined with m...
2,40248830,2,As social relationships are intertwined with m...
3,40248830,3,As social relationships are intertwined with m...
4,40248830,4,As social relationships are intertwined with m...
...,...,...,...
495,40933240,0,Autistic children have higher unintentional in...
496,40933240,1,Autistic children have higher unintentional in...
497,40933240,2,Autistic children have higher unintentional in...
498,40933240,3,Autistic children have higher unintentional in...


# Use LLM to generate synthetic questions

In [7]:
# set LLM handle
model_handle = 'llama3.2:1b'
print(model_handle)

llama3.2:1b


In [8]:
# make prompt template for synthetic quesitons
prompt_template = """
You are a layperson who is interested in autism spectrum disorders.
Write a general question about autism that can be answered by the ABSTRACT below.
Keep the question only one or two sentences long.
Just write the question itself; do not write anything else.

PASSAGE:
{abstract}
""".strip()
print(prompt_template)

You are a layperson who is interested in autism spectrum disorders.
Write a general question about autism that can be answered by the ABSTRACT below.
Keep the question only one or two sentences long.
Just write the question itself; do not write anything else.

PASSAGE:
{abstract}


In [9]:
def generate_question(abstract, seed):
    prompt_text = prompt_template.format(abstract=abstract)
    response = ollama.chat(model=model_handle, messages=[{'role' : 'user', 'content' : prompt_text}],
                          options={'seed' : seed})
    return response['message']['content'].strip()

In [10]:
# demo question generation
demo_abstract = df_synth.iloc[0]['abstract']
print(demo_abstract)
print()
print(generate_question(demo_abstract, seed=42))

As social relationships are intertwined with mental health recovery, it is important to address a client's social support network during mental health interventions. This seems even more important for autistic clients, because research suggests they have on average smaller networks and experience more loneliness than non-autistic individuals. Therefore, an interview assessing the social support network in relation to intervention goals was co-created together with stakeholders (autistic clients, mental healthcare professionals and a mother of an autistic client). In addition, the psychometric properties and acceptability of this Network-in-Action-Interview (NiA-I) were studied as pre-registered (AsPredicted #59767).

How do autism symptoms affect social relationships in individuals with autism?


In [11]:
def generate_questions_list(df):
    records = df.to_dict(orient='records')
    synthetic_questions = [generate_question(record['abstract'], record['ollama_seed']) \
    for record in tqdm(records)]
    return synthetic_questions

In [12]:
# generate synthetic
print(datetime.now())
df_synth['synthetic_question'] = generate_questions_list(df_synth)
print(datetime.now())

2025-09-12 01:28:38.895598


  0%|          | 0/500 [00:00<?, ?it/s]

2025-09-12 01:44:50.778779


In [13]:
# to demonstrate reproducibility, generate again
if False: # set to True to generate again and compare
    print(datetime.now())
    demo_reproducibility = pd.Series(generate_questions_list(df_synth))
    print(datetime.now())
    print((df_synth['synthetic_question']==demo_reproducibility).value_counts()) # all true if reproducible
    print(pd.concat([df_synth['synthetic_question'], demo_reproducibility], axis=1))

In [14]:
# finalize ground truth dataframe
df_synth = df_synth[['pmid', 'ollama_seed', 'synthetic_question']]
df_synth

,pmid,ollama_seed,synthetic_question
0,40248830,0,What factors do social support networks have o...
1,40248830,1,Can effective social relationships be fostered...
2,40248830,2,What are the key differences between the socia...
3,40248830,3,What factors contribute to increased social su...
4,40248830,4,How do social relationships impact the overall...
...,...,...,...
495,40933240,0,How can caregivers support autistic children's...
496,40933240,1,What challenges do parents of autistic childre...
497,40933240,2,How can autism-friendly environments be create...
498,40933240,3,How can parents with autistic children effecti...


In [15]:
# look at a few questions
print('\n'.join(df_synth['synthetic_question'].to_list()[:10]))

What factors do social support networks have on the mental health recovery process for individuals with autism?
Can effective social relationships be fostered through targeted intervention to address autism's unique challenges with mental health recovery?
What are the key differences between the social support networks of autistic individuals and those without autism?
What factors contribute to increased social support networks in individuals with autism?
How do social relationships impact the overall well-being and recovery of individuals with autism spectrum disorders?
What can clinicians do to improve the accuracy of predicting which children with autism may eventually develop intellectual disabilities?
Will future research aim to develop more accurate assessments of individuals at risk for autism spectrum disorders?
What role do researchers believe play in developing effective screening tests for identifying individuals at risk of autism spectrum disorder?
What can be done to impro

# Write CSV

In [16]:
# write CSV file
df_synth.to_csv('../data/data-synth-question.csv', index=False, sep='\t')

In [17]:
print(datetime.now())

2025-09-12 01:44:50.839899
